In [23]:
import asyncio
import csv
import json
import logging
import pandas as pd
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import time
import nest_asyncio
from pathlib import Path
import ast

# Enable nested event loops for Jupyter
nest_asyncio.apply()

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Import your existing conversation generator
import sys
import torch
import copy
import os

# Automatically add the root project directory to sys.path
project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
from simulators.conversation_simulator import ConversationConfig, MultiTurnConversationGenerator


class DPODatasetPipeline:
    """3-step pipeline for efficient DPO dataset generation"""
    
    def __init__(self, base_config: ConversationConfig, user_prompt_template_path: str, 
                 terminal_signal: str = "[[TERMINATE CHAT]]"):
        self.base_config = base_config
        self.terminal_signal = terminal_signal
        
        # Load the user meta prompt template
        if user_prompt_template_path != "":
            with open(user_prompt_template_path, 'r') as f:
                self.user_prompt_template = f.read()
    
    def load_csv_data(self, csv_path: str) -> pd.DataFrame:
        """Load and parse the CSV data"""
        df = pd.read_csv(csv_path)
        
        # Parse the conversation column (assuming it's stored as string representation of list)
        def parse_conversation(conv_str):
            try:
                # Handle the conversation string - it might be a JSON string or Python literal
                if isinstance(conv_str, str):
                    return ast.literal_eval(conv_str)
                return conv_str
            except (ValueError, SyntaxError) as e:
                logger.warning(f"Failed to parse conversation: {e}")
                return []

        df['conversation_parsed'] = df['conversation'].apply(parse_conversation)
        return df

    # =================== STEP 1: Generate User Queries ===================
    async def step1_generate_user_queries(self, df: pd.DataFrame, chunk_size: int = 30) -> List[Dict]:
        """Step 1: Generate initial user queries by calling UserSimulator directly"""
        logger.info(f"🔥 STEP 1: Generating initial user queries for {len(df)} examples")
        
        total_chunks = (len(df) + chunk_size - 1) // chunk_size
        all_queries = []
        
        # Initialize UserSimulator once
        from simulators.user_simulator import UserSimulator
        user_sim = UserSimulator(**self.base_config.user_generation_kwargs)
        
        for chunk_idx in range(total_chunks):
            start_idx = chunk_idx * chunk_size
            end_idx = min((chunk_idx + 1) * chunk_size, len(df))
            df_chunk = df.iloc[start_idx:end_idx]
            
            logger.info(f"Processing chunk {chunk_idx + 1}/{total_chunks} (rows {start_idx}-{end_idx-1})")
            
            # Process each row in the chunk
            chunk_queries = []
            for idx, row in df_chunk.iterrows():
                try:
                    # Create conversation text for context
                    conv_text = ""
                    for msg in row['conversation_parsed']:
                        role = msg['role']
                        content = msg['content']
                        if role == 'user':
                            conv_text += f"User: {content}\n"
                        elif role == 'assistant':
                            conv_text += f"Assistant: {content}\n"
                    
                    
                    # Use the existing user_prompt_template
                    formatted_user_prompt = self.user_prompt_template.format(
                        conversation=conv_text.strip(),
                        ground_truth=row['ground_truth'],
                        terminal_signal=self.terminal_signal,
                        chat_history=''
                    )
                    
                    
                    # Create temporary UserSimulator with the formatted prompt
                    temp_user_sim = UserSimulator(
                        user_meta_prompt=formatted_user_prompt,
                        **self.base_config.user_generation_kwargs
                    )
                    
                    # Generate the initial user query directly
                    user_query = await temp_user_sim.async_call([])
                    
                    if not user_query:
                        user_query = "I'm looking for movie recommendations."
                    
                    query_data = {
                        'id': row.get('dialog_id', idx),
                        'ground_truth': row['ground_truth'],
                        'original_conversation': row['conversation_parsed'],
                        'initial_user_query': user_query.strip()
                    }
                    chunk_queries.append(query_data)
                    
                except Exception as e:
                    logger.error(f"Error generating query for row {idx}: {e}")
                    # Add fallback query
                    query_data = {
                        'id': row.get('dialog_id', idx),
                        'ground_truth': row['ground_truth'],
                        'original_conversation': row['conversation_parsed'],
                        'initial_user_query': "I'm looking for movie recommendations."
                    }
                    chunk_queries.append(query_data)
            
            all_queries.extend(chunk_queries)
            logger.info(f"✅ Chunk {chunk_idx + 1} completed: {len(chunk_queries)} user queries generated")
        
        logger.info(f"🎉 STEP 1 COMPLETE: {len(all_queries)} user queries generated")
        return all_queries

    def save_user_queries(self, queries: List[Dict], output_path: str):
        """Save user queries to CSV"""
        df_queries = pd.DataFrame(queries)
        # Convert lists to JSON strings for CSV storage
        df_queries['original_conversation'] = df_queries['original_conversation'].apply(json.dumps)
        df_queries.to_csv(output_path, index=False)
        logger.info(f"💾 User queries saved to {output_path}")
    

    # =================== STEP 2: Generate Assistant Responses ===================
    async def step2_generate_assistant_responses(self, queries_csv_path: str, 
                                                num_responses: int = 2, chunk_size: int = 30) -> List[Dict]:
        """Step 2: Generate multiple assistant responses by calling AssistantSimulator directly"""
        logger.info(f"🔥 STEP 2: Generating {num_responses} assistant responses per query")
        
        # Load user queries
        df_queries = pd.read_csv(queries_csv_path)
        df_queries['original_conversation'] = df_queries['original_conversation'].apply(json.loads)
        
        total_chunks = (len(df_queries) + chunk_size - 1) // chunk_size
        all_response_pairs = []
        
        # Initialize AssistantSimulator once based on configuration
        if self.base_config.use_bedrock_assistant:
            from simulators.assistant_simulator import AssistantSimulator
            assistant_sim = AssistantSimulator(
                assistant_meta_prompt=self.base_config.assistant_meta_prompt,
                **self.base_config.assistant_generation_kwargs
            )
            logger.info("Using Bedrock AssistantSimulator")
        else:
            from simulators.local_assistant_simulator import LoRAAssistantSimulator
            assistant_sim = LoRAAssistantSimulator(
                assistant_meta_prompt=self.base_config.assistant_meta_prompt,
                lora_model_path=self.base_config.local_model_path,
                base_model_path=self.base_config.base_model_path,
                num_gpus=torch.cuda.device_count() if torch.cuda.is_available() else 1,
                **self.base_config.assistant_generation_kwargs
            )
            logger.info("Using Local LoRA AssistantSimulator")
        
        for chunk_idx in range(total_chunks):
            start_idx = chunk_idx * chunk_size
            end_idx = min((chunk_idx + 1) * chunk_size, len(df_queries))
            df_chunk = df_queries.iloc[start_idx:end_idx]
            
            logger.info(f"Processing chunk {chunk_idx + 1}/{total_chunks} (rows {start_idx}-{end_idx-1})")
            
            # Generate responses for each query in the chunk
            chunk_pairs = []
            for idx, row in df_chunk.iterrows():
                initial_query = row['initial_user_query']
                
                # Create conversation context with just the user query
                conversation_context = [{"role": "user", "content": initial_query}]
                
                # Generate multiple assistant responses directly
                for resp_idx in range(num_responses):
                    try:
                        if self.base_config.use_bedrock_assistant:
                            # Call Bedrock assistant directly
                            assistant_response = await assistant_sim.async_call(conversation_context)
                        else:
                            # Call Local LoRA assistant directly
                            assistant_response = assistant_sim.generate_response(conversation_context)
                        
                        if not assistant_response:
                            assistant_response = "I'd be happy to help you with movie recommendations! Could you tell me what genres you enjoy?"
                        
                        # Create response pair
                        pair_data = {
                            'id': row['id'],
                            'ground_truth': row['ground_truth'],
                            'original_conversation': row['original_conversation'],
                            'initial_user_query': initial_query,
                            'response_id': resp_idx,
                            'assistant_response': assistant_response.strip(),
                            'conversation_start': [
                                {"role": "user", "content": initial_query},
                                {"role": "assistant", "content": assistant_response.strip()}
                            ]
                        }
                        chunk_pairs.append(pair_data)
                        
                    except Exception as e:
                        logger.error(f"Error generating assistant response {resp_idx} for row {idx}: {e}")
                        # Add fallback response
                        pair_data = {
                            'id': row['id'],
                            'ground_truth': row['ground_truth'],
                            'original_conversation': row['original_conversation'],
                            'initial_user_query': initial_query,
                            'response_id': resp_idx,
                            'assistant_response': "I'd be happy to help you with movie recommendations! Could you tell me what genres you enjoy?",
                            'conversation_start': [
                                {"role": "user", "content": initial_query},
                                {"role": "assistant", "content": "I'd be happy to help you with movie recommendations! Could you tell me what genres you enjoy?"}
                            ]
                        }
                        chunk_pairs.append(pair_data)
            
            all_response_pairs.extend(chunk_pairs)
            logger.info(f"✅ Chunk {chunk_idx + 1} completed: {len(chunk_pairs)} response pairs generated")
        
        logger.info(f"🎉 STEP 2 COMPLETE: {len(all_response_pairs)} response pairs generated")
        return all_response_pairs

    def save_response_pairs(self, response_pairs: List[Dict], output_path: str):
        """Save response pairs to CSV with assistant_response_1 and assistant_response_2 columns"""
        # Group response pairs by id to combine responses into single rows
        grouped_responses = {}
        
        for pair in response_pairs:
            pair_id = pair['id']
            if pair_id not in grouped_responses:
                grouped_responses[pair_id] = {
                    'id': pair['id'],
                    'ground_truth': pair['ground_truth'],
                    'original_conversation': pair['original_conversation'],
                    'initial_user_query': pair['initial_user_query'],
                    'assistant_response_1': None,
                    'assistant_response_2': None,
                    'conversation_start_1': None,
                    'conversation_start_2': None
                }
            
            # Store responses based on response_id
            response_id = pair['response_id']
            if response_id == 0:
                grouped_responses[pair_id]['assistant_response_1'] = pair['assistant_response']
                grouped_responses[pair_id]['conversation_start_1'] = pair['conversation_start']
            elif response_id == 1:
                grouped_responses[pair_id]['assistant_response_2'] = pair['assistant_response']
                grouped_responses[pair_id]['conversation_start_2'] = pair['conversation_start']
        
        # Convert to DataFrame
        df_pairs = pd.DataFrame(list(grouped_responses.values()))
        
        # Convert lists to JSON strings for CSV storage
        df_pairs['original_conversation'] = df_pairs['original_conversation'].apply(json.dumps)
        df_pairs['conversation_start_1'] = df_pairs['conversation_start_1'].apply(json.dumps)
        df_pairs['conversation_start_2'] = df_pairs['conversation_start_2'].apply(json.dumps)
        
        df_pairs.to_csv(output_path, index=False)
        logger.info(f"💾 Response pairs saved to {output_path}")
        
        # Print summary
        total_pairs = len(grouped_responses)
        complete_pairs = len([p for p in grouped_responses.values() if p['assistant_response_1'] and p['assistant_response_2']])
        
        logger.info(f"📊 Response Pairs Summary:")
        logger.info(f"  Total pairs: {total_pairs}")
        logger.info(f"  Complete pairs (both responses): {complete_pairs} ({complete_pairs/total_pairs:.1%})")
        logger.info(f"  Columns: id, ground_truth, original_conversation, initial_user_query, assistant_response_1, assistant_response_2")

    
    # =================== STEP 3: Generate Full Conversations ===================
    async def step3_generate_full_conversations(self, pairs_csv_path: str, 
                                              num_samples: int = 3, chunk_size: int = 30) -> List[Dict]:
        """Step 3: Generate full conversations using existing fast batch processing"""
        logger.info(f"🔥 STEP 3: Generating {num_samples} full conversations per response pair")
        
        # Load response pairs (now in the new format)
        df_pairs = pd.read_csv(pairs_csv_path)
        df_pairs['original_conversation'] = df_pairs['original_conversation'].apply(json.loads)
        df_pairs['conversation_start_1'] = df_pairs['conversation_start_1'].apply(json.loads)
        df_pairs['conversation_start_2'] = df_pairs['conversation_start_2'].apply(json.loads)
        
        total_chunks = (len(df_pairs) + chunk_size - 1) // chunk_size
        all_final_data = []
        
        for chunk_idx in range(total_chunks):
            start_idx = chunk_idx * chunk_size
            end_idx = min((chunk_idx + 1) * chunk_size, len(df_pairs))
            df_chunk = df_pairs.iloc[start_idx:end_idx]
            
            logger.info(f"Processing chunk {chunk_idx + 1}/{total_chunks} (rows {start_idx}-{end_idx-1})")
            
            # Prepare batch for conversation generation - now handling both response variants
            all_prompts = []
            all_batch_configs = []
            
            for idx, row in df_chunk.iterrows():
                # Create conversation text for user prompt context
                conv_text = ""
                for msg in row['original_conversation']:
                    role = msg['role']
                    content = msg['content']
                    if role == 'user':
                        conv_text += f"User: {content}\n"
                    elif role == 'assistant':
                        conv_text += f"Assistant: {content}\n"
                
                # Fill in the template for user simulation
                custom_user_prompt = self.user_prompt_template.format(
                    conversation=conv_text.strip(),
                    ground_truth=row['ground_truth'],
                    terminal_signal=self.terminal_signal,
                    chat_history="{chat_history}"
                )
                
                # Generate conversations for both assistant responses
                for response_id in [1, 2]:  # response_1 and response_2
                    assistant_response = row[f'assistant_response_{response_id}']
                    conversation_start = row[f'conversation_start_{response_id}']
                    
                    # Skip if response is missing
                    if pd.isna(assistant_response) or not assistant_response:
                        continue
                    
                    # Create multiple samples for this response
                    for sample_idx in range(num_samples):
                        all_batch_configs.append({
                            'user_meta_prompt': custom_user_prompt,
                            'user_generation_kwargs': self.base_config.user_generation_kwargs,
                            'pair_id': f"{row['id']}_response_{response_id}_sample_{sample_idx}",
                            'original_id': row['id'],
                            'response_id': response_id,
                            'sample_id': sample_idx,
                            'ground_truth': row['ground_truth'],
                            'original_conversation': row['original_conversation'],
                            'initial_user_query': row['initial_user_query'],
                            'assistant_response': assistant_response,
                            'conversation_start': conversation_start
                        })
                        
                        # Start the conversation with the existing two turns
                        conversation_start_prompt = f"Continue this conversation:\nUser: {row['initial_user_query']}\nAssistant: {assistant_response}\n\nUser: "
                        all_prompts.append(conversation_start_prompt)
            
            logger.info(f"  → Generating {len(all_prompts)} conversations in parallel...")
            
            # Use existing fast batch generation
            custom_config = copy.deepcopy(self.base_config)
            generator = MultiTurnConversationGenerator(custom_config)
            
            conversations = await generator.generate_conversations_batch(
                prompts=all_prompts,
                conv_num=len(all_prompts),
                batch_configs=all_batch_configs
            )
            
            # Process results and group by original_id
            chunk_results = {}
            for config, generated_conv in zip(all_batch_configs, conversations):
                original_id = config['original_id']
                
                if original_id not in chunk_results:
                    chunk_results[original_id] = {
                        'id': original_id,
                        'ground_truth': config['ground_truth'],
                        'original_conversation': config['original_conversation'],
                        'initial_user_query': config['initial_user_query'],
                        'assistant_response_1': None,  # Will be filled from the first response_1 config
                        'assistant_response_2': None,  # Will be filled from the first response_2 config
                    }
                
                # Store assistant responses (only need to do this once per response type)
                response_id = config['response_id']
                if response_id == 1 and chunk_results[original_id]['assistant_response_1'] is None:
                    chunk_results[original_id]['assistant_response_1'] = config['assistant_response']
                elif response_id == 2 and chunk_results[original_id]['assistant_response_2'] is None:
                    chunk_results[original_id]['assistant_response_2'] = config['assistant_response']
                
                # Start with the fixed two turns
                conversation_start = config['conversation_start']
                
                if generated_conv and len(generated_conv) > 0:
                    # Combine: [user_query, assistant_response] + [generated_continuation]
                    full_conversation = conversation_start + generated_conv
                else:
                    # Fallback to just the start
                    full_conversation = conversation_start
                
                # Store conversation with proper naming
                sample_id = config['sample_id']
                conversation_key = f"response_{response_id}_conversation_{sample_id}"
                
                chunk_results[original_id][conversation_key] = full_conversation
                chunk_results[original_id][f"response_{response_id}_length_{sample_id}"] = len(full_conversation)
            
            # Convert to final format
            for result in chunk_results.values():
                all_final_data.append(result)
            
            logger.info(f"✅ Chunk {chunk_idx + 1} completed: {len(conversations)} conversations generated")
        
        logger.info(f"🎉 STEP 3 COMPLETE: {len(all_final_data)} final conversation groups generated")
        return all_final_data

    def save_final_dataset(self, final_data: List[Dict], output_path: str):
        """Save final DPO dataset to CSV with proper conversation columns"""
        df_final = pd.DataFrame(final_data)
        
        # Convert lists to JSON strings for CSV storage
        df_final['original_conversation'] = df_final['original_conversation'].apply(json.dumps)
        
        # Convert conversation columns to JSON (they have the format response_X_conversation_Y)
        for col in df_final.columns:
            if 'response_' in col and 'conversation_' in col and not col.endswith('_length_0') and not col.endswith('_length_1') and not col.endswith('_length_2'):
                df_final[col] = df_final[col].apply(lambda x: json.dumps(x) if x is not None else None)
        
        df_final.to_csv(output_path, index=False)
        logger.info(f"💾 Final DPO dataset saved to {output_path}")
        
        # Print summary
        total_entries = len(final_data)
        
        # Count successful conversations per response type
        response_1_success = 0
        response_2_success = 0
        
        for entry in final_data:
            for i in range(3):  # 3 samples per response
                if f"response_1_conversation_{i}" in entry and entry.get(f"response_1_length_{i}", 0) > 2:
                    response_1_success += 1
                if f"response_2_conversation_{i}" in entry and entry.get(f"response_2_length_{i}", 0) > 2:
                    response_2_success += 1
        
        total_conversations = total_entries * 2 * 3  # 2 responses × 3 samples each
        
        logger.info(f"📊 Final Dataset Summary:")
        logger.info(f"  Total entries: {total_entries}")
        logger.info(f"  Response 1 successful conversations: {response_1_success}")
        logger.info(f"  Response 2 successful conversations: {response_2_success}")
        logger.info(f"  Total successful conversations: {response_1_success + response_2_success}/{total_conversations}")
        logger.info(f"  Success rate: {(response_1_success + response_2_success)/total_conversations:.1%}")
        
        # Show column structure
        conversation_cols = [col for col in df_final.columns if 'response_' in col and 'conversation_' in col and not 'length' in col]
        response_cols = [col for col in df_final.columns if col.startswith('assistant_response_')]
        logger.info(f"  Assistant response columns: {response_cols}")
        logger.info(f"  Conversation columns: {conversation_cols}")

In [25]:
class PipelineConfig:
    """Shared configuration for the entire DPO pipeline"""
    def __init__(self):
        # Dataset and model configuration
        self.dataset = "inspired"  # Change this for different datasets
        self.local_model_path_name = "test_epoch5_seed2"
        self.alg = "vanilla"
        self.chunk_size = 30
        
        # Generation parameters
        self.assistant_responses_per_query = 2
        self.conversation_samples_per_response = 3
        
        # File paths
        self.csv_input_path = f"../datasets/{self.dataset}/multiturn_form/train.csv"
        self.user_prompt_template_path = "../prompts/test_user_prompt.txt"
        self.output_dir = f"{self.dataset}/DPO_pipeline/"
        
        # Derived paths
        self.step1_output = f"{self.output_dir}step1_user_queries.csv"
        self.step2_output = f"{self.output_dir}step2_response_pairs.csv"
        self.step3_output = f"{self.output_dir}step3_final_dataset.csv"
        
        # Model paths
        self.local_model_path = f"/home/sagemaker-user/csbai/multiturn_rl/outputs/{self.alg}/{self.dataset}/{self.local_model_path_name}"
        self.base_model_path = "meta-llama/Llama-3.2-1B-Instruct"
        
        # Generation kwargs
        self.assistant_generation_kwargs = {
            "temperature": 0.8,
            "max_tokens": 512,
            "model": "us.meta.llama3-2-1b-instruct-v1:0",
            "num_retries": 50
        }
        
        self.user_generation_kwargs = {
            "model": "us.anthropic.claude-sonnet-4-20250514-v1:0",
            "temperature": 0.8,
            "max_tokens": 256,
            "num_retries": 50
        }
        
        # Other settings
        self.max_total_turns = 4
        self.max_gen_workers = 20
        self.enable_batching = True
        self.use_bedrock_assistant = (self.alg == "vanilla")

# Global pipeline configuration
PIPELINE_CONFIG = PipelineConfig()

# =================== MAIN PIPELINE FUNCTIONS ===================

async def run_step1_user_queries(config: PipelineConfig = None):
    """Run Step 1: Generate user queries"""
    if config is None:
        config = PIPELINE_CONFIG
    
    logger.info(f"🔥 STEP 1: Generating user queries for dataset '{config.dataset}' using {config.alg}")
    
    # Create config - Step 1 only needs user generation, but we need to set the flags correctly
    conv_config = ConversationConfig(
        assistant_meta_prompt="",  # Not needed for step 1
        user_meta_prompt="",  # Will be overridden
        user_generation_kwargs=config.user_generation_kwargs,
        # Important: Set bedrock flag for proper simulator selection
        use_bedrock_assistant=config.use_bedrock_assistant,
        # Also include other necessary configs even though we don't use assistant
        local_model_path=config.local_model_path,
        base_model_path=config.base_model_path,
        max_gen_workers=config.max_gen_workers,
        enable_batching=config.enable_batching
    )
    
    logger.info(f"Using {'Bedrock' if config.use_bedrock_assistant else 'Local LoRA'} for user simulation")
    
    pipeline = DPODatasetPipeline(conv_config, config.user_prompt_template_path)
    
    # Load data and generate queries
    logger.info(f"Loading data from: {config.csv_input_path}")
    df = pipeline.load_csv_data(config.csv_input_path)
    logger.info(f"Processing {len(df)} conversations in chunks of {config.chunk_size}")
    
    queries = await pipeline.step1_generate_user_queries(df, chunk_size=config.chunk_size)
    
    # Ensure output directory exists
    Path(config.output_dir).mkdir(parents=True, exist_ok=True)
    pipeline.save_user_queries(queries, config.step1_output)
    
    logger.info(f"✅ Step 1 complete: {len(queries)} user queries saved to {config.step1_output}")
    return queries

In [26]:
def create_custom_config(dataset="inspired", alg="vanilla", local_model_path_name="test_epoch5_seed2", 
                        chunk_size=30, **kwargs):
    """Create a custom pipeline configuration"""
    config = PipelineConfig()
    config.dataset = dataset
    config.alg = alg
    config.local_model_path_name = local_model_path_name
    config.chunk_size = chunk_size
    
    # Update any additional parameters
    for key, value in kwargs.items():
        if hasattr(config, key):
            setattr(config, key, value)
        else:
            logger.warning(f"Unknown configuration parameter: {key}")
    
    # Recalculate derived paths
    config.csv_input_path = f"../datasets/{config.dataset}/multiturn_form/train.csv"
    config.output_dir = f"{config.dataset}/DPO_pipeline/"
    config.step1_output = f"{config.output_dir}step1_user_queries.csv"
    config.step2_output = f"{config.output_dir}step2_response_pairs.csv"
    config.step3_output = f"{config.output_dir}step3_final_dataset.csv"
    config.local_model_path = f"/home/sagemaker-user/csbai/multiturn_rl/outputs/{config.alg}/{config.dataset}/{config.local_model_path_name}"
    config.use_bedrock_assistant = (config.alg == "vanilla")
    
    return config

custom_config = create_custom_config(
    dataset="inspired", 
    alg="vanilla", 
    local_model_path_name="test_epoch5_seed2",
    chunk_size=30,
    assistant_responses_per_query=2,
    conversation_samples_per_response=3
)
await run_step1_user_queries(custom_config)

INFO: 🔥 STEP 1: Generating user queries for dataset 'inspired' using vanilla
INFO: Using Bedrock for user simulation
INFO: Loading data from: ../datasets/inspired/multiturn_form/train.csv
INFO: Processing 801 conversations in chunks of 30
INFO: 🔥 STEP 1: Generating initial user queries for 801 examples
INFO: Processing chunk 1/27 (rows 0-29)
INFO: ✅ Chunk 1 completed: 30 user queries generated
INFO: Processing chunk 2/27 (rows 30-59)
INFO: ✅ Chunk 2 completed: 30 user queries generated
INFO: Processing chunk 3/27 (rows 60-89)
Exception ignored in: <function UserSimulator.__del__ at 0x7f9690475cf0>
Traceback (most recent call last):
  File "/home/sagemaker-user/csbai/multiturn_rl/simulators/user_simulator.py", line 121, in __del__
    self._executor.shutdown(wait=True)
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/concurrent/futures/thread.py", line 235, in shutdown
    t.join()
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/threading.py", line 109

[{'id': '20191127-210600_875_live.pkl',
  'ground_truth': 'Knives Out',
  'original_conversation': [{'role': 'assistant',
    'content': 'Hi There! What types of movies do you like to watch?'},
   {'role': 'user',
    'content': "Hello! I'm more of an action movie or a good romance and mystery movie."},
   {'role': 'assistant',
    'content': 'I just saw the trailer for Knives Out when I went to see Joker and it looked like a good mix of action and mystery!'},
   {'role': 'user',
    'content': 'I seen that one too as I seen Joker about a month ago. I thought about asking my fiance about going and seeing it.'},
   {'role': 'assistant',
    'content': 'It looks like a good movie for people who like many different movies. It also has a great cast! I was surprised to see Chris Evans in the trailer!'},
   {'role': 'user',
    'content': "Maybe with Chris Evans in it it'll be easier to convince my fiance to see it. Do you know who else is in the cast?"},
   {'role': 'assistant',
    'conten

In [18]:
async def run_step2_assistant_responses(config: PipelineConfig = None):
    """Run Step 2: Generate assistant responses"""
    if config is None:
        config = PIPELINE_CONFIG
    
    logger.info(f"🔥 STEP 2: Generating {config.assistant_responses_per_query} assistant responses per query using {config.alg}")
    
    conv_config = ConversationConfig(
        assistant_meta_prompt="You are a helpful movie recommendation assistant. Provide personalized movie suggestions based on user preferences and engage in natural conversation about movies.",
        user_meta_prompt="",  # Not needed for step 2
        local_model_path=config.local_model_path,
        base_model_path=config.base_model_path,
        assistant_generation_kwargs=config.assistant_generation_kwargs,
        use_bedrock_assistant=config.use_bedrock_assistant,
        max_gen_workers=config.max_gen_workers,
        enable_batching=config.enable_batching
    )
    
    logger.info(f"Using {'Bedrock' if config.use_bedrock_assistant else 'Local LoRA'} for assistant generation")
    
    pipeline = DPODatasetPipeline(conv_config, "")  # Don't need user template for step 2
    
    logger.info(f"Loading user queries from: {config.step1_output}")
    response_pairs = await pipeline.step2_generate_assistant_responses(
        config.step1_output, 
        num_responses=config.assistant_responses_per_query, 
        chunk_size=config.chunk_size
    )
    
    pipeline.save_response_pairs(response_pairs, config.step2_output)
    
    logger.info(f"✅ Step 2 complete: {len(response_pairs)} response pairs saved to {config.step2_output}")
    return response_pairs

await run_step2_assistant_responses(custom_config)  

INFO: 🔥 STEP 2: Generating 2 assistant responses per query using vanilla
INFO: Using Bedrock for assistant generation
INFO: Loading user queries from: inspired/DPO_pipeline/step1_user_queries.csv
INFO: 🔥 STEP 2: Generating 2 assistant responses per query
INFO: Using Bedrock AssistantSimulator
INFO: Processing chunk 1/2 (rows 0-4)
INFO: ✅ Chunk 1 completed: 10 response pairs generated
INFO: Processing chunk 2/2 (rows 5-9)
INFO: ✅ Chunk 2 completed: 10 response pairs generated
INFO: 🎉 STEP 2 COMPLETE: 20 response pairs generated
INFO: 💾 Response pairs saved to inspired/DPO_pipeline/step2_response_pairs.csv
INFO: 📊 Response Pairs Summary:
INFO:   Total pairs: 10
INFO:   Complete pairs (both responses): 10 (100.0%)
INFO:   Columns: id, ground_truth, original_conversation, initial_user_query, assistant_response_1, assistant_response_2
INFO: ✅ Step 2 complete: 20 response pairs saved to inspired/DPO_pipeline/step2_response_pairs.csv


[{'id': '20191127-210600_875_live.pkl',
  'ground_truth': 'Knives Out',
  'original_conversation': [{'role': 'assistant',
    'content': 'Hi There! What types of movies do you like to watch?'},
   {'role': 'user',
    'content': "Hello! I'm more of an action movie or a good romance and mystery movie."},
   {'role': 'assistant',
    'content': 'I just saw the trailer for Knives Out when I went to see Joker and it looked like a good mix of action and mystery!'},
   {'role': 'user',
    'content': 'I seen that one too as I seen Joker about a month ago. I thought about asking my fiance about going and seeing it.'},
   {'role': 'assistant',
    'content': 'It looks like a good movie for people who like many different movies. It also has a great cast! I was surprised to see Chris Evans in the trailer!'},
   {'role': 'user',
    'content': "Maybe with Chris Evans in it it'll be easier to convince my fiance to see it. Do you know who else is in the cast?"},
   {'role': 'assistant',
    'conten

In [21]:
async def run_step3_full_conversations(config: PipelineConfig = None):
    """Run Step 3: Generate full conversations"""
    if config is None:
        config = PIPELINE_CONFIG
    
    logger.info(f"🔥 STEP 3: Generating {config.conversation_samples_per_response} conversations per response pair using {config.alg}")
    
    conv_config = ConversationConfig(
        assistant_meta_prompt="You are a helpful movie recommendation assistant. Provide personalized movie suggestions based on user preferences and engage in natural conversation about movies.",
        user_meta_prompt="",  # Will be overridden per conversation
        max_total_turns=config.max_total_turns,
        max_gen_workers=config.max_gen_workers,
        local_model_path=config.local_model_path,
        base_model_path=config.base_model_path,
        assistant_generation_kwargs=config.assistant_generation_kwargs,
        user_generation_kwargs=config.user_generation_kwargs,
        enable_batching=config.enable_batching,
        use_bedrock_assistant=config.use_bedrock_assistant
    )
    
    logger.info(f"Using {'Bedrock' if config.use_bedrock_assistant else 'Local LoRA'} for both user and assistant generation")
    
    pipeline = DPODatasetPipeline(conv_config, config.user_prompt_template_path)
    
    logger.info(f"Loading response pairs from: {config.step2_output}")
    final_data = await pipeline.step3_generate_full_conversations(
        config.step2_output, 
        num_samples=config.conversation_samples_per_response, 
        chunk_size=config.chunk_size
    )
    
    pipeline.save_final_dataset(final_data, config.step3_output)
    
    logger.info(f"✅ Step 3 complete: {len(final_data)} conversations saved to {config.step3_output}")
    return final_data

await run_step3_full_conversations(custom_config)

INFO: 🔥 STEP 3: Generating 3 conversations per response pair using vanilla
INFO: Using Bedrock for both user and assistant generation
INFO: Loading response pairs from: inspired/DPO_pipeline/step2_response_pairs.csv
INFO: 🔥 STEP 3: Generating 3 full conversations per response pair
INFO: Processing chunk 1/2 (rows 0-4)
INFO:   → Generating 30 conversations in parallel...


🤖 Initializing conversation generator...
✅ Using Bedrock Assistant Simulator
✅ Conversation generator ready!
🚀 Starting batch generation for 30 conversations...
🚀 Starting batch generation for 30 conversations with custom configs...
🔄 Round 1: Processing 30 active conversations
🔄 Round 2: Processing 30 active conversations
  🛑 Conversation 1 terminated by user
🔄 Round 3: Processing 29 active conversations
  🛑 Conversation 2 terminated by user
  🛑 Conversation 5 terminated by user
🔄 Round 4: Processing 27 active conversations
🔄 Round 5: Processing 27 active conversations
ERROR: Can't invoke 'us.anthropic.claude-sonnet-4-20250514-v1:0'. Reason: An error occurred (ThrottlingException) when calling the InvokeModel operation (reached max retries: 4): Too many tokens, please wait before trying again.


INFO: ✅ Chunk 1 completed: 30 conversations generated
INFO: Processing chunk 2/2 (rows 5-9)
INFO:   → Generating 30 conversations in parallel...


  🛑 Conversation 0 terminated by user
  🛑 Conversation 4 terminated by user
  🛑 Conversation 24 terminated by user
✅ Batch generation complete: 30/30 successful conversations
📊 Conversation lengths - Min: 5, Max: 11, Avg: 10.5
⏹️  3 conversations terminated early
✅ Batch generation complete: 30/30 successful conversations
🤖 Initializing conversation generator...
✅ Using Bedrock Assistant Simulator
✅ Conversation generator ready!
🚀 Starting batch generation for 30 conversations...
🚀 Starting batch generation for 30 conversations with custom configs...
🔄 Round 1: Processing 30 active conversations
  🛑 Conversation 27 terminated by user
🔄 Round 2: Processing 29 active conversations
  🛑 Conversation 2 terminated by user
  🛑 Conversation 25 terminated by user
  🛑 Conversation 26 terminated by user
🔄 Round 3: Processing 26 active conversations
🔄 Round 4: Processing 26 active conversations
  🛑 Conversation 29 terminated by user
🔄 Round 5: Processing 25 active conversations


INFO: ✅ Chunk 2 completed: 30 conversations generated
INFO: 🎉 STEP 3 COMPLETE: 10 final conversation groups generated
INFO: 💾 Final DPO dataset saved to inspired/DPO_pipeline/step3_final_dataset.csv
INFO: 📊 Final Dataset Summary:
INFO:   Total entries: 10
INFO:   Response 1 successful conversations: 30
INFO:   Response 2 successful conversations: 30
INFO:   Total successful conversations: 60/60
INFO:   Success rate: 100.0%
INFO:   Assistant response columns: ['assistant_response_1', 'assistant_response_2']
INFO:   Conversation columns: ['response_1_conversation_0', 'response_1_conversation_1', 'response_1_conversation_2', 'response_2_conversation_0', 'response_2_conversation_1', 'response_2_conversation_2']
INFO: ✅ Step 3 complete: 10 conversations saved to inspired/DPO_pipeline/step3_final_dataset.csv


  🛑 Conversation 15 terminated by user
✅ Batch generation complete: 30/30 successful conversations
📊 Conversation lengths - Min: 3, Max: 11, Avg: 10.1
⏹️  4 conversations terminated early
✅ Batch generation complete: 30/30 successful conversations


[{'id': '20191127-210600_875_live.pkl',
  'ground_truth': 'Knives Out',
  'original_conversation': [{'role': 'assistant',
    'content': 'Hi There! What types of movies do you like to watch?'},
   {'role': 'user',
    'content': "Hello! I'm more of an action movie or a good romance and mystery movie."},
   {'role': 'assistant',
    'content': 'I just saw the trailer for Knives Out when I went to see Joker and it looked like a good mix of action and mystery!'},
   {'role': 'user',
    'content': 'I seen that one too as I seen Joker about a month ago. I thought about asking my fiance about going and seeing it.'},
   {'role': 'assistant',
    'content': 'It looks like a good movie for people who like many different movies. It also has a great cast! I was surprised to see Chris Evans in the trailer!'},
   {'role': 'user',
    'content': "Maybe with Chris Evans in it it'll be easier to convince my fiance to see it. Do you know who else is in the cast?"},
   {'role': 'assistant',
    'conten